Status: exploratory

Boundary effects in spatial-domain (Leiden) clustering

Follows the sc-best-practices.org spatial-domains tutorial's \"combined graph\" recipe (expression-neighbor graph + spatial-neighbor graph, joint Leiden), then asks: does a `bosperrus`-style border-effect correction of the PCA embedding (as a function of distance to the nearest border spot) change the resulting clusters, and does that change count as an *improvement*?

**Reads:** `sq.datasets.visium_hne_adata()` (squidpy's bundled adult mouse brain Visium demo, ~2700 spots; ships with an approximate ground-truth `cluster` annotation from the original 10x/squidpy processing — not manual pathologist ground truth, but a reasonable proxy for evaluation).

**Writes:** `../../results/exploratory/boundary_effects_clustering/` (fit_quality.csv, ground_truth_concordance.csv, concordance_by_distance_bin.csv, boundary_signal_diagnostic.csv, distance_by_cluster_boxplot.svg). 

**Runtime:** a few minutes on this small demo dataset.

**Border-spot definition:** a spot is a border spot if it has fewer than 6 neighbors in the Visium hex-grid spatial graph — this also flags spots adjacent to internal tissue holes/folds, not just the outer capture-area perimeter, which is the correct generalization of \"fewer neighbours = boundary artifact\" from the bosperrus paper's centrality-measure setting."

In [ ]:
%load_ext autoreload
%autoreload 2

import pathlib

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse
from scipy.stats import kruskal, spearmanr, fisher_exact
from scipy.spatial.distance import cdist
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    homogeneity_completeness_v_measure,
    silhouette_samples,
)

from bosperrus import Flow
from bosperrus.distances import distance_to_pointset
from bosperrus.fit import ConstantFit, PiecewiseLinearFit, ExponentialSaturationFit, MichaelisMentenFit

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=80, facecolor="white")

RNG_SEED = 0
N_PCS = 30
ALPHA = 0.2  # weight of the spatial graph in the joint graph, matches the tutorial's example
N_DISTANCE_BINS = 4

N_TOP_GENES = 2000       # HVGs for the shared preprocessing pipeline (§1 baseline, stage 1, stage 2)
ALPHA_GRID = [0.0, 0.1, 0.2, 0.3, 0.5]   # swept in §7's stage x alpha comparison
N_PILOT_GENES = 100      # stage-2 DE-on vs DE-off correctness pilot
PILOT_AGREEMENT_THRESHOLD = 0.9
PILOT_B_CORRELATION_THRESHOLD = 0.9

OUT_DIR = pathlib.Path("../../results/exploratory/boundary_effects_clustering")
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Shared helpers\n\nUsed by the baseline (§1) and by all three correction stages (§7), so every stage is compared under one controlled preprocessing pipeline instead of squidpy's shipped one.

In [ ]:
def run_standard_preprocessing(raw_counts, obs_names, var_names, n_top_genes=N_TOP_GENES, n_pcs=N_PCS):
    """raw_counts: spots x genes raw UMI counts (sparse or dense).
    normalize_total -> log1p -> highly_variable_genes -> PCA. Returns the resulting
    AnnData (subset to the HVGs), so callers can read both .obsm["X_pca"] and
    .X (the normalized+log1p'd HVG expression matrix) from the same object."""
    X = raw_counts.tocsr().copy() if sparse.issparse(raw_counts) else np.asarray(raw_counts).copy()
    tmp = ad.AnnData(X)
    tmp.obs_names = obs_names
    tmp.var_names = var_names
    sc.pp.normalize_total(tmp)
    sc.pp.log1p(tmp)
    sc.pp.highly_variable_genes(tmp, n_top_genes=n_top_genes, flavor="seurat")
    tmp = tmp[:, tmp.var["highly_variable"]].copy()
    sc.pp.pca(tmp, n_comps=n_pcs, random_state=RNG_SEED)
    return tmp


def pca_from_expression(expression_matrix, n_pcs=N_PCS):
    """expression_matrix: spots x genes, already normalized/log1p'd/HVG-restricted."""
    tmp = ad.AnnData(np.asarray(expression_matrix))
    sc.pp.pca(tmp, n_comps=n_pcs, random_state=RNG_SEED)
    return tmp.obsm["X_pca"]


def expression_connectivities(pca_basis):
    """Expression-neighbor connectivity graph from an arbitrary PCA basis."""
    tmp = ad.AnnData(np.zeros((pca_basis.shape[0], 1)))
    tmp.obsm["X_pca"] = pca_basis
    sc.pp.neighbors(tmp, use_rep="X_pca", random_state=RNG_SEED)
    return tmp.obsp["connectivities"]


def leiden_labels(adjacency, n_obs, random_state=RNG_SEED):
    """Leiden clustering directly from a precomputed adjacency (joint) graph."""
    tmp = ad.AnnData(np.zeros((n_obs, 1)))
    sc.tl.leiden(tmp, adjacency=adjacency, key_added="leiden", random_state=random_state)
    return tmp.obs["leiden"].values


def zone_stratified_means(values, zone_labels):
    """Mean of `values` overall ("all") and within each unique zone label."""
    means = {"all": float(np.nanmean(values))}
    for zone in np.unique(zone_labels):
        means[zone] = float(np.nanmean(np.asarray(values)[zone_labels == zone]))
    return means


class FastPiecewiseLinearFit(PiecewiseLinearFit):
    """PiecewiseLinearFit with differential_evolution refinement disabled --
    verified to reproduce identical fitted params/corrected values on synthetic
    data at ~5-6x the speed, needed for tractability at gene-scale (~2000 columns)."""
    def fit(self, refine_fit=False):
        super().fit(refine_fit=refine_fit)

## 1. Load data + baseline joint-graph clustering (no correction)\n\nReplicates `spatial_domains.ipynb`'s combined-graph recipe: expression-neighbor graph + Visium hex-grid spatial-neighbor graph, joint Leiden. This is the \"before correction\" baseline everything else compares against. As of §7, the baseline is built through `run_standard_preprocessing()` from raw counts (`adata.raw.X`) rather than reusing squidpy's shipped PCA/HVGs, so that every correction stage is compared under one controlled pipeline.

In [ ]:
adata = sq.datasets.visium_hne_adata()

raw_counts = adata.raw.X.tocsr().astype(np.float64)
baseline_pp = run_standard_preprocessing(raw_counts, adata.obs_names, adata.raw.var_names)
adata.obsm["X_pca"] = baseline_pp.obsm["X_pca"]
hvg_gene_names = baseline_pp.var_names.to_numpy()
hvg_expression_baseline = np.asarray(
    baseline_pp.X.todense() if sparse.issparse(baseline_pp.X) else baseline_pp.X
)

sc.pp.neighbors(adata, n_pcs=N_PCS, random_state=RNG_SEED)
nn_graph_genes = adata.obsp["connectivities"]

sq.gr.spatial_neighbors(adata, coord_type="grid", n_neighs=6)
nn_graph_space = adata.obsp["spatial_connectivities"]

joint_graph_before = (1 - ALPHA) * nn_graph_genes + ALPHA * nn_graph_space
sc.tl.leiden(adata, adjacency=joint_graph_before, key_added="leiden_before", random_state=RNG_SEED)

print(f"HVGs selected: {hvg_expression_baseline.shape[1]}")
print(adata.obs["leiden_before"].value_counts())

2. Border-spot detection + distance to nearest border spot

A spot is a border spot if its degree in the hex-grid spatial graph is < 6 (fewer than the full complement of hex neighbors — either at the outer tissue edge or next to an internal hole/fold). 
Distance is then computed with the existing `bosperrus.distances.distance_to_pointset` — nearest-neighbor distance to that set of border-spot coordinates.

In [ ]:
spatial_degree = np.asarray((nn_graph_space > 0).sum(axis=1)).ravel()
is_border_spot = spatial_degree < 6

coords = adata.obsm["spatial"]
border_coords = coords[is_border_spot]

dist_to_border = distance_to_pointset(coords, border_coords)
adata.obs["distance_to_border"] = dist_to_border.values
adata.obs["is_border_spot"] = is_border_spot

print(f"{is_border_spot.sum()} / {adata.n_obs} spots ({is_border_spot.mean():.1%}) flagged as border spots (spatial degree < 6)")
adata.obs["distance_to_border"].describe()

3. BOSPERRUS correction of the top PCs\n\nRather than correcting per-gene (thousands of noisy independent fits), each of the top `N_PCS` principal components is treated as a `bosperrus` \"score\" and fit against `distance_to_border`, exactly like the package already does for centrality measures — `ConstantFit`/`PiecewiseLinearFit`/`ExponentialSaturationFit`/`MichaelisMentenFit`, AIC-selected, additively corrected toward the fitted plateau. `flow.flow()` is called with no keyword arguments so this cell works whether the environment resolves the currently-installed PyPI `bosperrus` (`measures=`/`calculate_rel_ll_to_baseline=`) or the in-progress rename (`score_names=`/`baseline_fit_class=`) — both accept the all-defaults call.

In [ ]:
pc_cols = [f"PC{i}" for i in range(N_PCS)]
# bosperrus>=0.2.3 fixes a bug where Flow.flow() returned all-NaN corrected columns for a
# non-default (e.g. obs_names) index, so this can now use adata.obs_names directly.
pcs_df = pd.DataFrame(adata.obsm["X_pca"][:, :N_PCS], columns=pc_cols, index=adata.obs_names)
distances_series = pd.Series(adata.obs["distance_to_border"].values, index=adata.obs_names, name="distance_to_border")

flow = Flow.from_distances_and_scores(distances=distances_series, scores=pcs_df)
flow.flow()

fit_quality = flow.fit_quality.T[["best_fit_type", "observed_effect_strength", "observed_half_life", "affected samples"]]
fit_quality.to_csv(OUT_DIR / "fit_quality.csv")
fit_quality

In [ ]:
corrected_cols = [f"BOSPERRUS corrected {c}" for c in pc_cols]
adata.obsm["X_pca_corrected"] = flow.observations[corrected_cols].values

n_constant = (fit_quality["best_fit_type"] == "Constant Fit").sum()
print(f"{n_constant} / {N_PCS} PCs selected the null (Constant Fit) model — i.e. no detectable boundary effect")

### 3a. A model-derived border/interior split, from the piecewise-linear elbow\n\nThe hex-degree definition used above is topological (only the outermost rim, ~10% of spots) and independent of whether the fitted model actually detected an effect that large. `PiecewiseLinearFit`'s breakpoint `b` is a more direct, statistically-grounded alternative: it's literally the fitted distance at which *that PC's* boundary effect plateaus. Below, `b` is pulled per-PC (only for PCs where `PiecewiseLinearFit` won) to see how far into the tissue each one actually extends.

In [ ]:
piecewise_pcs = fit_quality[fit_quality["best_fit_type"] == "Piecewise Linear Fit"].index.tolist()
d_max = adata.obs["distance_to_border"].max()

elbow_table = pd.DataFrame([
    {
        "pc": pc,
        "elbow_b": flow.best_fits[pc].params["piecewise_linear_b"],
        "elbow_frac_of_dmax": flow.best_fits[pc].params["piecewise_linear_b"] / d_max,
        "affected_fraction": fit_quality.loc[pc, "affected samples"],
    }
    for pc in piecewise_pcs
]).sort_values("elbow_b").reset_index(drop=True)

elbow_table.to_csv(OUT_DIR / "piecewise_linear_elbows.csv", index=False)
elbow_table

The elbows span almost the entire observed range, and split into two groups with a fairly clean gap in `affected_fraction` between them:\n\n- **Localized** (elbow well inside the tissue, `affected_fraction` roughly ≤ 0.6): e.g. PC20/PC19/PC5 plateau within ~7-10% of the max distance. These look like genuine, thin boundary artifacts.\n- **Global-gradient** (`affected_fraction` ≥ ~0.85, up to 0.9996): PC0/PC1/PC16 in particular never plateau at all within the data — their fitted breakpoint sits at essentially 100% of the max distance. A \"boundary effect\" that never saturates across an entire tissue section is a strong hint that `PiecewiseLinearFit` is fitting a real radial anatomical gradient (e.g. cortex-to-center depth), not a rim artifact — exactly the biology/artifact confound flagged earlier. **Using PC0's own elbow would be a bad choice of cutoff** for that reason, even though it's the highest-variance PC.\n\nDefine the model-derived border zone from the *localized* group's outer envelope (the largest elbow among PCs below the gap), rather than from any single PC.

In [ ]:
LOCALIZED_AFFECTED_FRACTION_MAX = 0.7  # natural gap in the table sits between ~0.61 and ~0.85

localized = elbow_table[elbow_table["affected_fraction"] < LOCALIZED_AFFECTED_FRACTION_MAX]
elbow_cutoff = localized["elbow_b"].max()
cutoff_pc = localized.loc[localized["elbow_b"].idxmax(), "pc"]

is_border_spot_elbow = adata.obs["distance_to_border"].values <= elbow_cutoff
adata.obs["is_border_spot_elbow"] = is_border_spot_elbow

print(f"Elbow-derived cutoff: {elbow_cutoff:.1f} ({elbow_cutoff / d_max:.1%} of max distance), set by {cutoff_pc}")
print(f"Topological definition:   {is_border_spot.sum()} / {adata.n_obs} spots ({is_border_spot.mean():.1%})")
print(f"Elbow-derived definition: {is_border_spot_elbow.sum()} / {adata.n_obs} spots ({is_border_spot_elbow.mean():.1%})")
print(f"Overlap: {(is_border_spot & is_border_spot_elbow).sum()} spots flagged by both")

4. Rebuild the joint graph from corrected PCs, re-cluster
Same recipe as step 1 (same `ALPHA`, same spatial graph), but the expression-neighbor graph is now built on `X_pca_corrected` instead of `X_pca`.

In [ ]:
sc.pp.neighbors(adata, use_rep="X_pca_corrected", key_added="corrected", random_state=RNG_SEED)
nn_graph_genes_corrected = adata.obsp["corrected_connectivities"]

joint_graph_after = (1 - ALPHA) * nn_graph_genes_corrected + ALPHA * nn_graph_space
sc.tl.leiden(adata, adjacency=joint_graph_after, key_added="leiden_after", random_state=RNG_SEED)

print(adata.obs["leiden_after"].value_counts())
print(f"\nn_clusters before={adata.obs['leiden_before'].nunique()}, after={adata.obs['leiden_after'].nunique()}")
print("(cluster *count* differs between runs at fixed resolution regardless of correction quality; "
      "keep this in mind when reading the metrics below rather than assuming a like-for-like partition size.)")

## 5. Evaluation — did clustering improve?\n\nThree ground-truth-free diagnostics, all targeted at the *specific* boundary-artifact hypothesis rather than generic cluster tightness (which can't distinguish \"removed an artifact\" from \"threw away real biology\"):\n\n1. **Boundary-signal diagnostic** — Kruskal-Wallis + eta² between cluster identity and `distance_to_border`. A real correction should shrink this association.\n2. **Spatial-neighbor label purity**, border vs. interior gap.\n3. **Silhouette score**, border vs. interior gap.\n\nAll three ask the same underlying question without needing any external label: does a spot's proximity to the tissue edge predict its cluster assignment or how well it fits that cluster, more than it should if the correction were doing its job? A fourth section compares against the dataset's shipped `cluster` annotation, but that one is **not** treated as ground truth (see the caveat markdown before it) — it's a secondary reference, not primary evidence.

In [ ]:
def cluster_distance_association(cluster_labels, distances):
    """Kruskal-Wallis association between cluster identity and distance-to-border,
    plus its eta-squared effect size: (H - k + 1) / (n - k)."""
    groups = [distances[cluster_labels == c] for c in pd.unique(cluster_labels)]
    stat, p = kruskal(*groups)
    k, n = len(groups), len(distances)
    eta_sq = (stat - k + 1) / (n - k)
    return stat, p, eta_sq

h_before, p_before, eta_before = cluster_distance_association(
    adata.obs["leiden_before"].values, adata.obs["distance_to_border"].values
)
h_after, p_after, eta_after = cluster_distance_association(
    adata.obs["leiden_after"].values, adata.obs["distance_to_border"].values
)

boundary_signal = pd.DataFrame(
    {"before_correction": [h_before, p_before, eta_before], "after_correction": [h_after, p_after, eta_after]},
    index=["kruskal_H", "p_value", "eta_squared"],
)
boundary_signal.to_csv(OUT_DIR / "boundary_signal_diagnostic.csv")
boundary_signal

### 5a-continued. Two more ground-truth-free structural diagnostics\n\nBoth are targeted at the *specific* boundary-artifact hypothesis (not generic cluster tightness, which can't distinguish \"removed an artifact\" from \"threw away real biology\"). Both compare a **border-vs-interior gap** within each clustering, rather than an absolute score — the claim to test is that border spots look structurally worse than interior spots before correction, and that gap should shrink after.\n\n- **Spatial-neighbor label purity**: for each spot, the fraction of its hex-grid spatial neighbors sharing its cluster label. A border spot mislabeled purely because of a technical artifact should disagree with its physical neighbors more often than an interior spot with the same true biology would.\n- **Silhouette score** in the embedding actually used for that clustering (raw `X_pca` for `leiden_before`, `X_pca_corrected` for `leiden_after`): does a border spot look like it belongs to its assigned cluster as much as an interior spot does?

In [ ]:
def neighbor_label_purity(adjacency, labels):
    """Fraction of each node's spatial neighbors sharing its cluster label."""
    labels = np.asarray(labels)
    adj = adjacency.tocsr()
    purity = np.full(adj.shape[0], np.nan)
    for i in range(adj.shape[0]):
        neighbors = adj.indices[adj.indptr[i]:adj.indptr[i + 1]]
        if len(neighbors) > 0:
            purity[i] = np.mean(labels[neighbors] == labels[i])
    return purity

adata.obs["neighbor_purity_before"] = neighbor_label_purity(nn_graph_space, adata.obs["leiden_before"].values)
adata.obs["neighbor_purity_after"] = neighbor_label_purity(nn_graph_space, adata.obs["leiden_after"].values)

def border_interior_gap(values, is_border):
    interior_mean = np.nanmean(values[~is_border])
    border_mean = np.nanmean(values[is_border])
    return interior_mean, border_mean, interior_mean - border_mean

purity_rows = []
for label, col in [("before_correction", "neighbor_purity_before"), ("after_correction", "neighbor_purity_after")]:
    interior_mean, border_mean, gap = border_interior_gap(adata.obs[col].values, is_border_spot)
    purity_rows.append({"interior_spots": interior_mean, "border_spots": border_mean, "gap (interior - border)": gap})

purity_summary = pd.DataFrame(purity_rows, index=["before_correction", "after_correction"]).T
purity_summary.to_csv(OUT_DIR / "neighbor_purity_border_vs_interior.csv")
purity_summary

In [ ]:
sil_before = silhouette_samples(adata.obsm["X_pca"][:, :N_PCS], adata.obs["leiden_before"].values)
sil_after = silhouette_samples(adata.obsm["X_pca_corrected"], adata.obs["leiden_after"].values)

silhouette_rows = []
for label, values in [("before_correction", sil_before), ("after_correction", sil_after)]:
    interior_mean, border_mean, gap = border_interior_gap(values, is_border_spot)
    silhouette_rows.append({"interior_spots": interior_mean, "border_spots": border_mean, "gap (interior - border)": gap})

silhouette_summary = pd.DataFrame(silhouette_rows, index=["before_correction", "after_correction"]).T
silhouette_summary.to_csv(OUT_DIR / "silhouette_border_vs_interior.csv")
silhouette_summary

### 5a-continued (2). Same two diagnostics, using the elbow-derived border zone instead of the topological one\n\nRe-runs the neighbor-purity and silhouette border-vs-interior gaps from `is_border_spot_elbow` (§3a) rather than the hex-degree rim. If the topological definition was too narrow to capture what the model actually corrected, this should show a clearer (or at least different) signal.

In [ ]:
def gap_row(phase, values, border_mask):
    interior_mean, border_mean, gap = border_interior_gap(values, border_mask)
    return {"phase": phase, "interior": interior_mean, "border": border_mean, "gap (interior - border)": gap}

purity_comparison = pd.DataFrame(
    [
        {"definition": "topological (degree<6)", **gap_row("before_correction", adata.obs["neighbor_purity_before"].values, is_border_spot)},
        {"definition": "topological (degree<6)", **gap_row("after_correction", adata.obs["neighbor_purity_after"].values, is_border_spot)},
        {"definition": "elbow-derived", **gap_row("before_correction", adata.obs["neighbor_purity_before"].values, is_border_spot_elbow)},
        {"definition": "elbow-derived", **gap_row("after_correction", adata.obs["neighbor_purity_after"].values, is_border_spot_elbow)},
    ]
)
purity_comparison.to_csv(OUT_DIR / "neighbor_purity_border_definitions_comparison.csv", index=False)
purity_comparison

In [ ]:
silhouette_comparison = pd.DataFrame(
    [
        {"definition": "topological (degree<6)", **gap_row("before_correction", sil_before, is_border_spot)},
        {"definition": "topological (degree<6)", **gap_row("after_correction", sil_after, is_border_spot)},
        {"definition": "elbow-derived", **gap_row("before_correction", sil_before, is_border_spot_elbow)},
        {"definition": "elbow-derived", **gap_row("after_correction", sil_after, is_border_spot_elbow)},
    ]
)
silhouette_comparison.to_csv(OUT_DIR / "silhouette_border_definitions_comparison.csv", index=False)
silhouette_comparison

### 5b. Concordance with `cluster` — NOT a ground truth, read with caution\n\n`adata.obs[\"cluster\"]` was itself produced by a standard graph-based clustering pipeline on this same dataset's (uncorrected) counts. It is **not** an independent ground truth for whether the boundary correction helped — agreeing with it more just means agreeing with someone else's border-affected clustering more, and *disagreeing* with it near the border is exactly what a working correction might do if that annotation has the same artifact baked in. Keep this section as a secondary reference point, not the primary evidence — the ground-truth-free diagnostics above (eta², neighbor purity, silhouette gap) are the ones this analysis actually leans on.

In [ ]:
def concordance(true_labels, pred_labels):
    hom, comp, vmeas = homogeneity_completeness_v_measure(true_labels, pred_labels)
    return {
        "ARI": adjusted_rand_score(true_labels, pred_labels),
        "NMI": normalized_mutual_info_score(true_labels, pred_labels),
        "homogeneity": hom,
        "completeness": comp,
        "v_measure": vmeas,
    }

ground_truth = adata.obs["cluster"].values
metrics_before = concordance(ground_truth, adata.obs["leiden_before"].values)
metrics_after = concordance(ground_truth, adata.obs["leiden_after"].values)

comparison = pd.DataFrame({"before_correction": metrics_before, "after_correction": metrics_after})
comparison["delta"] = comparison["after_correction"] - comparison["before_correction"]
comparison.to_csv(OUT_DIR / "ground_truth_concordance.csv")
comparison

In [ ]:
adata.obs["distance_bin"] = pd.qcut(
    adata.obs["distance_to_border"], N_DISTANCE_BINS, labels=[f"Q{i+1}_of_{N_DISTANCE_BINS}" for i in range(N_DISTANCE_BINS)]
)  # Q1 = closest to a border spot, Q{N} = deepest in the tissue interior

rows = []
for b in adata.obs["distance_bin"].cat.categories:
    mask = (adata.obs["distance_bin"] == b).values
    m_before = concordance(ground_truth[mask], adata.obs["leiden_before"].values[mask])
    m_after = concordance(ground_truth[mask], adata.obs["leiden_after"].values[mask])
    rows.append({
        "distance_bin": b,
        "n_spots": int(mask.sum()),
        "ARI_before": m_before["ARI"], "ARI_after": m_after["ARI"], "ARI_delta": m_after["ARI"] - m_before["ARI"],
        "NMI_before": m_before["NMI"], "NMI_after": m_after["NMI"], "NMI_delta": m_after["NMI"] - m_before["NMI"],
    })

stratified = pd.DataFrame(rows)
stratified.to_csv(OUT_DIR / "concordance_by_distance_bin.csv", index=False)
stratified

## 6. Visualization

Spatial maps (ground truth vs. before vs. after) and, for each clustering, a boxplot of `distance_to_border` per cluster — a cluster that's mostly a thin rim hugging the tissue edge should show up as a box sitting much lower than the others, and that box should either vanish or rise toward the rest after correction.

In [ ]:
sq.pl.spatial_scatter(adata, color=["cluster", "leiden_before", "leiden_after"], wspace=0.4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
sns.boxplot(data=adata.obs, x="leiden_before", y="distance_to_border", ax=axes[0])
axes[0].set_title("before correction")
sns.boxplot(data=adata.obs, x="leiden_after", y="distance_to_border", ax=axes[1])
axes[1].set_title("after correction")
plt.tight_layout()
plt.savefig(OUT_DIR / "distance_by_cluster_boxplot.svg")
plt.show()

## 7. Where should the correction be applied? Comparing three correction targets\n\nEverything above corrects the top PCs (stage 3 below). Two mechanistically different alternatives: correct **total counts per spot** (stage 1 — a single scalar per spot; can only fix a *uniform-depth* capture artifact, e.g. fewer molecules captured per spot near a cut edge) and correct **per-gene expression** (stage 2 — up to `N_TOP_GENES` independent fits; can additionally fix a *compositional* shift, specific genes/programs behaving differently near the edge). Comparing all three under one shared preprocessing pipeline (§ \"Shared helpers\") and a small `ALPHA_GRID` sweep tells us whether the correction target matters, and if stage 1 vs. stage 2 differ, that itself is evidence about which failure mode dominates here. No plots in this section — numbers/tables only.

### 7.0 Three-way zone classification (frozen from the baseline, reused for every stage x alpha below)\n\nBuilt entirely from what's already computed in §2/§3a — no new distance or model fitting. `is_border_spot ⊆ is_border_spot_elbow` by construction, so this is a clean three-way split by set difference: `direct_border` = topological rim, `buffer_zone` = within the elbow but not on the rim, `interior` = beyond the elbow. Computed once here and never recomputed per stage — recomputing the elbow per stage would let each stage's own correction redefine the ruler measuring that correction's improvement.

In [ ]:
direct_border = is_border_spot
buffer_zone = is_border_spot_elbow & ~is_border_spot
interior = ~is_border_spot_elbow

zone_labels = np.where(direct_border, "direct_border", np.where(buffer_zone, "buffer_zone", "interior"))
adata.obs["zone"] = zone_labels

zone_counts = pd.Series(zone_labels).value_counts()
zone_counts.to_csv(OUT_DIR / "zone_classification_counts.csv")
zone_counts

### 7.1 Stage 1 — correct total counts per spot\n\nFits BOSPERRUS on `log1p(total_counts)` vs. `distance_to_border` (matches the established convention in `truncated_graphs/src/utils/visium_hd.py`, which fits on the log scale for the same Gaussian-residual reason bosperrus's AIC selection needs — raw per-spot totals are right-skewed and heteroscedastic). The correction is inverted back into a per-spot size factor and applied by rescaling each spot's raw count vector, then re-run through the shared preprocessing pipeline. `log1p(total_counts) >= 0` is a hard constraint of the real quantity, so the corrected value is clipped at 0 before `expm1` inversion — not an arbitrary defensive clip.

In [ ]:
raw_total = np.asarray(raw_counts.sum(axis=1)).ravel()
log1p_total = np.log1p(raw_total)
dist_series = pd.Series(adata.obs["distance_to_border"].values, index=adata.obs_names, name="distance_to_border")

flow_stage1 = Flow.from_distances_and_scores(
    distances=dist_series,
    scores=pd.DataFrame({"log1p_total_counts": log1p_total}, index=adata.obs_names),
)
flow_stage1.flow()

corrected_log1p = flow_stage1.observations["BOSPERRUS corrected log1p_total_counts"].values
n_spots_clipped = int((corrected_log1p < 0).sum())
corrected_total = np.expm1(np.clip(corrected_log1p, 0, None))

scale_factor = corrected_total / raw_total
rescaled_counts = raw_counts.multiply(scale_factor.reshape(-1, 1)).tocsr()

stage1_fit_quality = flow_stage1.fit_quality.T
stage1_fit_quality.to_csv(OUT_DIR / "stage1_total_counts_fit_quality.csv")
print(f"Stage 1: best fit = {stage1_fit_quality['best_fit_type'].iloc[0]}, "
      f"effect_strength = {stage1_fit_quality['observed_effect_strength'].iloc[0]:.3f}, "
      f"n_spots_clipped_at_zero = {n_spots_clipped}")

baseline_pp_stage1 = run_standard_preprocessing(rescaled_counts, adata.obs_names, adata.raw.var_names)
X_pca_stage1 = baseline_pp_stage1.obsm["X_pca"]
print(f"Stage 1 corrected PCA basis: {X_pca_stage1.shape}")

### 7.2 Stage 2 — correct per-gene expression (pilot, then full run)\n\nCorrecting `N_TOP_GENES` HVGs independently means `N_TOP_GENES` AIC-based model comparisons. `PiecewiseLinearFit`'s `differential_evolution` refinement (always on by default in `Flow.flow()`, no opt-out) makes this ~5-6x slower than necessary; `FastPiecewiseLinearFit` (defined in \"Shared helpers\") disables it. Both are already tractable (~6 vs ~28 min for 2000 genes, verified directly against the installed package on synthetic data), so this pilot isn't a timing gate — it's a **correctness check**: does skipping DE change which model wins or where the elbow lands on real, possibly-noisier gene curves (the synthetic verification used a clean signal, which may not generalize)? Run on `N_PILOT_GENES` genes both ways and compare.

In [ ]:
pilot_genes = hvg_gene_names[:N_PILOT_GENES]
pilot_expression = pd.DataFrame(
    hvg_expression_baseline[:, :N_PILOT_GENES], columns=pilot_genes, index=adata.obs_names
)

flow_pilot_de_on = Flow.from_distances_and_scores(distances=dist_series, scores=pilot_expression)
flow_pilot_de_on.flow()

flow_pilot_de_off = Flow.from_distances_and_scores(distances=dist_series, scores=pilot_expression)
flow_pilot_de_off.flow(fits=[ConstantFit, FastPiecewiseLinearFit, ExponentialSaturationFit, MichaelisMentenFit])

types_on = flow_pilot_de_on.fit_quality.T["best_fit_type"]
types_off = flow_pilot_de_off.fit_quality.T["best_fit_type"]
agreement = (types_on == types_off).mean()

both_piecewise = (types_on == "Piecewise Linear Fit") & (types_off == "Piecewise Linear Fit")
piecewise_genes = types_on.index[both_piecewise]
b_on = np.array([flow_pilot_de_on.best_fits[g].params["piecewise_linear_b"] for g in piecewise_genes])
b_off = np.array([flow_pilot_de_off.best_fits[g].params["piecewise_linear_b"] for g in piecewise_genes])
b_correlation = float(np.corrcoef(b_on, b_off)[0, 1]) if len(b_on) > 1 else np.nan

print(f"Pilot (n={N_PILOT_GENES} genes): AIC-winner agreement DE-on vs DE-off = {agreement:.1%}")
print(f"Both selected Piecewise Linear Fit for {both_piecewise.sum()} genes; "
      f"elbow (b) correlation between DE-on/DE-off = {b_correlation:.4f}")

In [ ]:
use_de_off = (agreement >= PILOT_AGREEMENT_THRESHOLD) and (
    np.isnan(b_correlation) or b_correlation >= PILOT_B_CORRELATION_THRESHOLD
)
fits_for_stage2 = (
    [ConstantFit, FastPiecewiseLinearFit, ExponentialSaturationFit, MichaelisMentenFit] if use_de_off else None
)
print(f"Pilot {'passed' if use_de_off else 'FAILED'} -> using "
      f"{'DE-off (fast)' if use_de_off else 'DE-on (default, full refinement)'} for the full "
      f"{N_TOP_GENES}-gene stage-2 fit.")

full_expression = pd.DataFrame(hvg_expression_baseline, columns=hvg_gene_names, index=adata.obs_names)
flow_stage2 = Flow.from_distances_and_scores(distances=dist_series, scores=full_expression)
flow_stage2.flow(fits=fits_for_stage2)

stage2_fit_quality = flow_stage2.fit_quality.T
stage2_fit_quality.to_csv(OUT_DIR / "stage2_gene_fit_quality.csv")
print(stage2_fit_quality["best_fit_type"].value_counts())

corrected_gene_cols = [f"BOSPERRUS corrected {g}" for g in hvg_gene_names]
corrected_hvg_expression = flow_stage2.observations[corrected_gene_cols].values
X_pca_stage2 = pca_from_expression(corrected_hvg_expression)
print(f"Stage 2 corrected PCA basis: {X_pca_stage2.shape}")

### 7.3 Stage 3 — PCA-transformed counts\n\nNo new cell needed: §3 already corrects the top `N_PCS` of `adata.obsm["X_pca"]`, and §1 now builds that PCA through the same shared `run_standard_preprocessing()` pipeline as stages 1/2. `adata.obsm["X_pca_corrected"]` (from §3) *is* stage 3 under the shared pipeline.

### 7.4 Alpha x stage sweep, silhouette by zone\n\nBOSPERRUS fitting (stages 1-3 above) depends only on `distances`/the corrected embedding, not on `ALPHA` — it's fit once per stage. `ALPHA` only enters at the joint-graph-build + Leiden step, so each stage's expression-neighbor connectivity is computed once and cached, then swept cheaply over `ALPHA_GRID`. `ALPHA=0.0` is a deliberate pure-expression control (isolates whether spatial-graph blending alone helps, independent of any BOSPERRUS correction); `ALPHA=0.2` is the value used everywhere else in this notebook.

In [ ]:
stage_bases = {
    "baseline": adata.obsm["X_pca"],
    "stage1_total_counts": X_pca_stage1,
    "stage2_per_gene": X_pca_stage2,
    "stage3_pca": adata.obsm["X_pca_corrected"],
}

rows = []
for stage_name, basis in stage_bases.items():
    conn = expression_connectivities(basis)
    for alpha in ALPHA_GRID:
        joint = (1 - alpha) * conn + alpha * nn_graph_space
        labels = leiden_labels(joint, n_obs=basis.shape[0])
        sil = silhouette_samples(basis, labels)
        zone_means = zone_stratified_means(sil, zone_labels)
        n_clusters = len(np.unique(labels))
        for zone, mean_val in zone_means.items():
            n_spots = len(zone_labels) if zone == "all" else int((zone_labels == zone).sum())
            rows.append({
                "stage": stage_name,
                "alpha": alpha,
                "zone": zone,
                "n_spots": n_spots,
                "n_clusters": n_clusters,
                "mean_silhouette": mean_val,
            })
    print(f"done: {stage_name}")

comparison_table = pd.DataFrame(rows)
comparison_table.to_csv(OUT_DIR / "stage_alpha_zone_silhouette_comparison.csv", index=False)
comparison_table

In [ ]:
# Readable slice: silhouette by stage x zone at alpha=0.2 (the value used everywhere
# else in this notebook) -- continuity check against §5's already-established
# silhouette-gap direction, even though absolute numbers shift under the rebuilt baseline.
pivot_at_alpha02 = (
    comparison_table[np.isclose(comparison_table["alpha"], 0.2)]
    .pivot(index="stage", columns="zone", values="mean_silhouette")
    [["direct_border", "buffer_zone", "interior", "all"]]
)
pivot_at_alpha02.to_csv(OUT_DIR / "stage_zone_silhouette_at_alpha_0.2.csv")
pivot_at_alpha02

## 8. Follow-up diagnostics — why does stage 1 win?\n\nNumbers only, no plots. All of these reuse objects already computed above (`flow`/`pcs_df` from §3, `flow_stage1`/`flow_stage2`, `hvg_expression_baseline`/`corrected_hvg_expression`, `stage_bases`, `zone_labels`) — nothing here refits BOSPERRUS.

In [ ]:
### 8.0 Elbow / half-life aggregation per strategy
# observed_half_life (not the raw model param `b`) is the comparable quantity across
# model types -- PiecewiseLinearFit's `b` is a distance breakpoint, but
# ExponentialSaturationFit's `b` is a *rate* (1/distance), so averaging raw `b` across
# model types would silently mix incompatible units. ConstantFit rows are excluded
# (trivially half_life=0, not "a small effect").

def summarize_strategy(fit_quality_df, strategy_name, n_total):
    detected = fit_quality_df[fit_quality_df["best_fit_type"] != "Constant Fit"]
    return {
        "strategy": strategy_name,
        "n_total": n_total,
        "n_detected": len(detected),
        "pct_detected": len(detected) / n_total,
        "mean_half_life": detected["observed_half_life"].mean(),
        "std_half_life": detected["observed_half_life"].std(),
        "mean_affected_fraction": detected["affected samples"].mean(),
        "std_affected_fraction": detected["affected samples"].std(),
    }

strategy_summary = pd.DataFrame([
    summarize_strategy(stage1_fit_quality, "stage1_total_counts", 1),
    summarize_strategy(stage2_fit_quality, "stage2_per_gene", N_TOP_GENES),
    summarize_strategy(fit_quality, "stage3_pca", N_PCS),
])
strategy_summary.to_csv(OUT_DIR / "strategy_elbow_halflife_summary.csv", index=False)
strategy_summary

In [ ]:
### 8.1 Top-10 most/least affected genes (stage 2), by effect strength and by half-life
# Ranked only among genes that got a non-null model -- ConstantFit genes all tie at 0,
# which isn't an interesting ranking. Ranking by raw |effect_strength| can surface
# numerically degenerate fits (denominator near zero) rather than genuinely strong
# effects -- shown as-is (not filtered) so any such entries are visible, not hidden.

detected_stage2 = stage2_fit_quality[stage2_fit_quality["best_fit_type"] != "Constant Fit"].copy()
detected_stage2["gene"] = detected_stage2.index
detected_stage2["is_mt"] = adata.raw.var.loc[detected_stage2.index, "mt"].values

cols = ["gene", "is_mt", "best_fit_type", "observed_effect_strength", "observed_half_life", "affected samples"]

top10_by_effect_strength = (
    detected_stage2.reindex(detected_stage2["observed_effect_strength"].abs().sort_values(ascending=False).index)
    .head(10)[cols]
)
bottom10_by_effect_strength = (
    detected_stage2.reindex(detected_stage2["observed_effect_strength"].abs().sort_values(ascending=True).index)
    .head(10)[cols]
)
# largest half-life = least localized ("global-gradient"-like, extends deep into tissue)
top10_by_half_life = detected_stage2.sort_values("observed_half_life", ascending=False).head(10)[cols]
# smallest half-life = most localized (sharp, thin, genuine-border-artifact-like)
bottom10_by_half_life = detected_stage2.sort_values("observed_half_life", ascending=True).head(10)[cols]

for name, df in [
    ("top10_by_effect_strength", top10_by_effect_strength),
    ("bottom10_by_effect_strength", bottom10_by_effect_strength),
    ("top10_by_half_life_least_localized", top10_by_half_life),
    ("bottom10_by_half_life_most_localized", bottom10_by_half_life),
]:
    df.to_csv(OUT_DIR / f"stage2_{name}.csv", index=False)

print("top10_by_effect_strength:")
print(top10_by_effect_strength.to_string(index=False))
print("\nbottom10_by_effect_strength:")
print(bottom10_by_effect_strength.to_string(index=False))
print("\ntop10_by_half_life (least localized):")
print(top10_by_half_life.to_string(index=False))
print("\nbottom10_by_half_life (most localized):")
print(bottom10_by_half_life.to_string(index=False))

In [ ]:
### 8.2 Are mitochondrial genes over-represented among the "detected effect" genes?
# If boundary-affected genes were disproportionately mt-genes, that would be fairly
# direct evidence for a technical (stress/dissociation-like) artifact rather than real
# biology -- cheap Fisher exact test on the 2x2 contingency table.

is_mt_all = adata.raw.var.loc[hvg_gene_names, "mt"].values
is_detected_all = (stage2_fit_quality["best_fit_type"] != "Constant Fit").values

contingency = np.array([
    [int(np.sum(is_mt_all & is_detected_all)), int(np.sum(is_mt_all & ~is_detected_all))],
    [int(np.sum(~is_mt_all & is_detected_all)), int(np.sum(~is_mt_all & ~is_detected_all))],
])
odds_ratio, p_value = fisher_exact(contingency)

mt_enrichment = pd.DataFrame(
    contingency, index=["mt_gene", "non_mt_gene"], columns=["detected_effect", "not_detected"]
)
mt_enrichment.to_csv(OUT_DIR / "mt_gene_enrichment_contingency.csv")
print(mt_enrichment)
print(f"\nn_mt_genes_in_HVGs = {int(is_mt_all.sum())} / {len(is_mt_all)}")
print(f"odds_ratio = {odds_ratio:.3f}, p_value = {p_value:.4f}")

In [ ]:
### 8.3 Does the gene-level correction, mapped through the ORIGINAL PCA loadings, agree
### with what the independent per-PC fit found?
# Stage 2 and stage 3 use DIFFERENT PCA bases (stage 2 re-derives new loadings from the
# corrected gene matrix; stage 3 additively corrects PCs from the original basis), so
# comparing X_pca_stage2 to X_pca_stage3 directly isn't meaningful -- different bases
# were never going to agree. The well-posed version: project the gene-level correction
# through the ORIGINAL (fixed) loadings and compare THAT to stage 3's actual per-PC shift.

V_original = baseline_pp.varm["PCs"]  # genes x n_pcs, aligned with hvg_gene_names order
gene_correction = corrected_hvg_expression - hvg_expression_baseline  # spots x genes
implied_pc_shift = gene_correction @ V_original  # spots x n_pcs
actual_pc_shift = adata.obsm["X_pca_corrected"] - adata.obsm["X_pca"]  # spots x n_pcs

per_pc_correlation = pd.Series(
    [np.corrcoef(implied_pc_shift[:, i], actual_pc_shift[:, i])[0, 1] for i in range(N_PCS)],
    index=pc_cols,
    name="implied_vs_actual_pc_shift_correlation",
)
overall_correlation = float(np.corrcoef(implied_pc_shift.ravel(), actual_pc_shift.ravel())[0, 1])
relative_frobenius_diff = float(
    np.linalg.norm(implied_pc_shift - actual_pc_shift) / np.linalg.norm(actual_pc_shift)
)

per_pc_correlation.to_csv(OUT_DIR / "gene_vs_pc_correction_consistency_per_pc.csv")
print(f"Overall correlation (implied gene-level shift vs actual per-PC shift): {overall_correlation:.4f}")
print(f"Relative Frobenius-norm difference: {relative_frobenius_diff:.4f}")
print("\nPer-PC correlation:")
print(per_pc_correlation.to_string())

In [ ]:
### 8.4 Cross-stage per-spot correction-magnitude consistency
# Per spot, how much did each stage's correction actually move it? If all three
# strategies are picking up a shared "how boundary-affected is this spot" latent trait,
# these three magnitude vectors should correlate; if they're fixing genuinely different
# things, they won't.

magnitude_stage1 = np.abs(np.clip(corrected_log1p, 0, None) - log1p_total)
magnitude_stage2 = np.linalg.norm(corrected_hvg_expression - hvg_expression_baseline, axis=1)
magnitude_stage3 = np.linalg.norm(adata.obsm["X_pca_corrected"] - adata.obsm["X_pca"], axis=1)

pairs = [("stage1", "stage2", magnitude_stage1, magnitude_stage2),
         ("stage1", "stage3", magnitude_stage1, magnitude_stage3),
         ("stage2", "stage3", magnitude_stage2, magnitude_stage3)]
rows = []
for name_a, name_b, a, b in pairs:
    rho, p = spearmanr(a, b)
    rows.append({"pair": f"{name_a}_vs_{name_b}", "spearman_rho": rho, "p_value": p})

magnitude_consistency = pd.DataFrame(rows)
magnitude_consistency.to_csv(OUT_DIR / "cross_stage_correction_magnitude_consistency.csv", index=False)
magnitude_consistency

In [ ]:
### 8.5 Silhouette improvement vs. baseline, by alpha x zone x stage
# Derived directly from §7.4's comparison_table -- "improvement" should be read off
# this delta, not off the raw values side by side.

baseline_lookup = (
    comparison_table[comparison_table["stage"] == "baseline"]
    .set_index(["alpha", "zone"])["mean_silhouette"]
)
delta_table = comparison_table[comparison_table["stage"] != "baseline"].copy()
delta_table["silhouette_delta_vs_baseline"] = delta_table.apply(
    lambda r: r["mean_silhouette"] - baseline_lookup.loc[(r["alpha"], r["zone"])], axis=1
)
delta_table.to_csv(OUT_DIR / "silhouette_delta_vs_baseline.csv", index=False)
delta_table.pivot_table(index=["stage", "alpha"], columns="zone", values="silhouette_delta_vs_baseline")

In [ ]:
### 8.6 Aggregate AIC-detection-strength per strategy
# Sums (baseline_AIC - best_AIC) across all units in a strategy -- weighted by how
# confidently each unit beat the null ConstantFit, not just a binary count of
# non-Constant selections. ConstantFit's own AIC is recomputed in closed form
# (matches Fit._fit_constant + evaluate_fit.log_likelihood/akaike_information_criterion)
# rather than re-instantiating 2000+30+1 ConstantFit objects.

def constant_baseline_aic(matrix):
    """matrix: N x M. Returns the ConstantFit AIC for each of the M columns."""
    N = matrix.shape[0]
    means = matrix.mean(axis=0)
    resid = matrix - means
    sigma2 = np.maximum((resid ** 2).mean(axis=0), np.finfo(float).tiny)
    log_likelihood = -0.5 * N * (np.log(2 * np.pi * sigma2) + 1)
    return 2 * 2 - 2 * log_likelihood  # num_params = len({"constant_c": c}) + 1 = 2

baseline_aic_stage1 = constant_baseline_aic(log1p_total.reshape(-1, 1))[0]
best_aic_stage1 = flow_stage1.best_fits["log1p_total_counts"].AIC
aic_gap_stage1 = np.array([baseline_aic_stage1 - best_aic_stage1])

baseline_aic_stage2 = constant_baseline_aic(full_expression.values)
best_aic_stage2 = np.array([flow_stage2.best_fits[g].AIC for g in hvg_gene_names])
aic_gap_stage2 = baseline_aic_stage2 - best_aic_stage2

baseline_aic_stage3 = constant_baseline_aic(pcs_df.values)
best_aic_stage3 = np.array([flow.best_fits[pc].AIC for pc in pc_cols])
aic_gap_stage3 = baseline_aic_stage3 - best_aic_stage3

aic_detection_summary = pd.DataFrame({
    "strategy": ["stage1_total_counts", "stage2_per_gene", "stage3_pca"],
    "n_units": [1, N_TOP_GENES, N_PCS],
    "total_aic_gap": [aic_gap_stage1.sum(), aic_gap_stage2.sum(), aic_gap_stage3.sum()],
    "mean_aic_gap": [aic_gap_stage1.mean(), aic_gap_stage2.mean(), aic_gap_stage3.mean()],
})
aic_detection_summary.to_csv(OUT_DIR / "aic_detection_strength_summary.csv", index=False)
aic_detection_summary

In [ ]:
### 8.7a Alpha=0.2 cluster labels for every stage (baseline/stage3 already exist as
### leiden_before/leiden_after; stage1/stage2 recomputed here -- cheap, since the
### expensive part (BOSPERRUS fitting) is already done and cached in X_pca_stage{1,2}).

adata.obs["leiden_baseline"] = adata.obs["leiden_before"].astype(str).astype("category")
adata.obs["leiden_stage3"] = adata.obs["leiden_after"].astype(str).astype("category")

conn_stage1_a02 = expression_connectivities(X_pca_stage1)
joint_stage1_a02 = (1 - ALPHA) * conn_stage1_a02 + ALPHA * nn_graph_space
adata.obs["leiden_stage1"] = pd.Categorical(leiden_labels(joint_stage1_a02, n_obs=X_pca_stage1.shape[0]))

conn_stage2_a02 = expression_connectivities(X_pca_stage2)
joint_stage2_a02 = (1 - ALPHA) * conn_stage2_a02 + ALPHA * nn_graph_space
adata.obs["leiden_stage2"] = pd.Categorical(leiden_labels(joint_stage2_a02, n_obs=X_pca_stage2.shape[0]))

stage_cluster_keys = {
    "baseline": "leiden_baseline",
    "stage1_total_counts": "leiden_stage1",
    "stage2_per_gene": "leiden_stage2",
    "stage3_pca": "leiden_stage3",
}
print({k: adata.obs[v].nunique() for k, v in stage_cluster_keys.items()})

In [ ]:
### 8.7b Rim-cluster identification (baseline only, frozen) via direct_border enrichment
# Flags the baseline cluster whose spot composition is most disproportionately
# direct_border (rarer, 10.5% dataset-wide, so a clearer signal than buffer_zone which
# is already 58% of all spots this run). A minimum-size filter avoids a tiny cluster
# trivially reading as "100% border" by chance.

MIN_CLUSTER_SIZE_FOR_RIM_CANDIDACY = 20
dataset_pct_direct_border = float((zone_labels == "direct_border").mean())

cluster_zone_df = adata.obs.groupby("leiden_baseline", observed=True).agg(
    n_spots=("zone", "size"),
    pct_direct_border=("zone", lambda s: (s == "direct_border").mean()),
)
rim_candidates = cluster_zone_df[cluster_zone_df["n_spots"] >= MIN_CLUSTER_SIZE_FOR_RIM_CANDIDACY]
rim_candidates = rim_candidates.sort_values("pct_direct_border", ascending=False)
rim_cluster_id = rim_candidates.index[0]

rim_mask = (adata.obs["leiden_baseline"] == rim_cluster_id).values
n_border_in_rim = int((zone_labels[rim_mask] == "direct_border").sum())
contingency_rim = np.array([
    [n_border_in_rim, int(rim_mask.sum()) - n_border_in_rim],
    [int((zone_labels == "direct_border").sum()) - n_border_in_rim,
     len(zone_labels) - int(rim_mask.sum()) - (int((zone_labels == "direct_border").sum()) - n_border_in_rim)],
])
rim_odds_ratio, rim_p_value = fisher_exact(contingency_rim)

print(f"Dataset-wide pct direct_border: {dataset_pct_direct_border:.1%}")
print(cluster_zone_df.sort_values('pct_direct_border', ascending=False).to_string())
print(f"\nCandidate rim cluster: {rim_cluster_id} "
      f"({rim_mask.sum()} spots, {rim_candidates.loc[rim_cluster_id, 'pct_direct_border']:.1%} direct_border)")
print(f"Enrichment vs. rest of dataset: odds_ratio={rim_odds_ratio:.2f}, p_value={rim_p_value:.2e}")
cluster_zone_df.to_csv(OUT_DIR / "baseline_cluster_zone_composition.csv")

In [ ]:
### 8.7c Spatial coherence via squidpy's permutation-null neighborhood enrichment
# A properly-controlled version of the earlier ad hoc neighbor-purity metric: for each
# stage's OWN clustering (not frozen -- clustering itself is what changes per stage),
# self-enrichment z-score per cluster (diagonal of the zscore matrix), averaged.
# Still doesn't resolve the deeper issue flagged earlier -- a spatially-smooth artifact
# and spatially-smooth real biology both look like genuine (non-chance) homophily here
# -- this is a better-controlled version of an intrinsically ambiguous signal, not a fix.

nhood_rows = []
for stage_name, cluster_key in stage_cluster_keys.items():
    zscore, _ = sq.gr.nhood_enrichment(
        adata, cluster_key=cluster_key, connectivity_key="spatial",
        n_perms=1000, seed=RNG_SEED, copy=True, show_progress_bar=False,
    )
    diag_z = np.diag(zscore)
    nhood_rows.append({
        "stage": stage_name,
        "n_clusters": len(diag_z),
        "mean_self_enrichment_z": float(np.mean(diag_z)),
        "median_self_enrichment_z": float(np.median(diag_z)),
    })

nhood_summary = pd.DataFrame(nhood_rows)
nhood_summary.to_csv(OUT_DIR / "nhood_enrichment_self_zscore_by_stage.csv", index=False)
nhood_summary

In [ ]:
### 8.8 E-distance between the rim cluster and its nearest neighbor, across stages
# "Cluster identity" isn't comparable across stages (different partitions altogether,
# not a relabeling of the same one) -- so this freezes the two SPOT SETS from the
# baseline clustering (the rim cluster identified in 8.7b, and whichever other baseline
# cluster is closest to it by E-distance) and just re-embeds those same fixed spots in
# each stage's own PCA basis. A real fix should shrink this specific pair's distance
# (the rim cluster's spots pulling toward their likely-true neighbor) without needing
# any cluster-identity matching across stages.

def e_distance(A, B):
    d_ab = cdist(A, B).mean()
    d_aa = cdist(A, A).mean()
    d_bb = cdist(B, B).mean()
    return 2 * d_ab - d_aa - d_bb

baseline_basis = adata.obsm["X_pca"]
other_cluster_ids = [c for c in adata.obs["leiden_baseline"].cat.categories if c != rim_cluster_id]
e_dist_to_others = {}
for c in other_cluster_ids:
    other_mask = (adata.obs["leiden_baseline"] == c).values
    if other_mask.sum() < 5:
        continue
    e_dist_to_others[c] = e_distance(baseline_basis[rim_mask], baseline_basis[other_mask])

neighbor_cluster_id = min(e_dist_to_others, key=e_dist_to_others.get)
neighbor_mask = (adata.obs["leiden_baseline"] == neighbor_cluster_id).values
print(f"Rim cluster {rim_cluster_id} ({rim_mask.sum()} spots) <-> "
      f"nearest baseline cluster {neighbor_cluster_id} ({neighbor_mask.sum()} spots), "
      f"baseline E-distance = {e_dist_to_others[neighbor_cluster_id]:.4f}")

edistance_rows = []
for stage_name, basis in stage_bases.items():
    d = e_distance(basis[rim_mask], basis[neighbor_mask])
    edistance_rows.append({"stage": stage_name, "e_distance_rim_vs_neighbor": d})

edistance_table = pd.DataFrame(edistance_rows)
edistance_table["e_distance_delta_vs_baseline"] = (
    edistance_table["e_distance_rim_vs_neighbor"] - edistance_table.loc[
        edistance_table["stage"] == "baseline", "e_distance_rim_vs_neighbor"
    ].iloc[0]
)
edistance_table.to_csv(OUT_DIR / "rim_neighbor_edistance_by_stage.csv", index=False)
edistance_table

## 9. Piecewise-linear fits, overlaid per strategy

One panel per strategy, every unit that selected `PiecewiseLinearFit` as its best model drawn as a semi-transparent line. Each curve is min-max normalized between its own value at `d=0` and `d=d_max` (0 at the border, 1 at its own asymptote) -- the underlying genes/PCs/total-counts differ in absolute scale by orders of magnitude, so a shared raw y-axis would be meaningless; normalizing makes the **elbow location** (the thing that actually distinguishes a localized artifact from a global gradient) directly comparable across strategies. This is exactly the visual counterpart of §8.0's `observed_half_life` summary.

### 9.1 All genes (no HVG restriction), mitochondrial genes highlighted

Direct follow-up to §8.2, which was inconclusive because only 3 of 13 mouse mt-genes made the 2000-HVG cut. Refits on the full gene set (minus genes with zero counts everywhere -- not an HVG-style selection, just removing genes with literally no signal to fit against distance at all) so all 13 mt-genes can be seen against the full background. Uses each gene's own AIC-selected model, not piecewise-only -- 2 of the 3 previously-seen mt-genes picked Michaelis-Menten, so restricting to piecewise-only here would hide most of them. This is a ~9x larger fit than stage 2's 2000 genes (expect roughly an hour with DE off), so the result is cached to CSV -- re-running this cell after the first time just reloads it.

In [ ]:
all_genes_cache_path = OUT_DIR / "all_genes_fit_quality.csv"
CHUNK_SIZE = 2000  # matches stage 2's already-proven-safe scale

if all_genes_cache_path.exists():
    all_genes_fit_quality = pd.read_csv(all_genes_cache_path, index_col=0)
    print(f"Loaded cached fit_quality for {len(all_genes_fit_quality)} genes from {all_genes_cache_path}")
else:
    import gc

    tmp_all_genes = ad.AnnData(raw_counts.copy())
    tmp_all_genes.obs_names = adata.obs_names
    tmp_all_genes.var_names = adata.raw.var_names
    sc.pp.filter_genes(tmp_all_genes, min_counts=1)
    sc.pp.normalize_total(tmp_all_genes)
    sc.pp.log1p(tmp_all_genes)
    all_gene_names = tmp_all_genes.var_names.to_numpy()
    all_gene_expression = np.asarray(
        tmp_all_genes.X.todense() if sparse.issparse(tmp_all_genes.X) else tmp_all_genes.X
    )
    n_genes = len(all_gene_names)
    print(f"Fitting {n_genes} / {raw_counts.shape[1]} genes "
          f"(dropped {raw_counts.shape[1] - n_genes} genes with zero counts everywhere) "
          f"in chunks of {CHUNK_SIZE} to bound peak memory -- a single ~18k-gene Flow call "
          f"previously died with an out-of-memory kernel crash (~4 live Fit objects per "
          f"gene, each holding several length-N arrays, adds up fast at this scale).")

    fit_quality_chunks = []
    n_chunks = -(-n_genes // CHUNK_SIZE)
    for chunk_i, start in enumerate(range(0, n_genes, CHUNK_SIZE)):
        chunk_names = all_gene_names[start:start + CHUNK_SIZE]
        chunk_df = pd.DataFrame(
            all_gene_expression[:, start:start + CHUNK_SIZE], columns=chunk_names, index=adata.obs_names
        )
        chunk_flow = Flow.from_distances_and_scores(distances=dist_series, scores=chunk_df)
        chunk_flow.flow(fits=[ConstantFit, FastPiecewiseLinearFit, ExponentialSaturationFit, MichaelisMentenFit])
        fit_quality_chunks.append(chunk_flow.fit_quality.T.copy())
        del chunk_flow, chunk_df
        gc.collect()
        print(f"  chunk {chunk_i + 1}/{n_chunks} done ({start}-{start + len(chunk_names)})")

    all_genes_fit_quality = pd.concat(fit_quality_chunks)
    all_genes_fit_quality.to_csv(all_genes_cache_path)

print(all_genes_fit_quality["best_fit_type"].value_counts())

mt_flag = adata.raw.var["mt"].reindex(all_genes_fit_quality.index).fillna(False).astype(bool)
all_genes_fit_quality["is_mt"] = mt_flag.values
print(f"mt genes present in fitted set: {int(mt_flag.sum())} / {int(adata.raw.var['mt'].sum())}")


In [ ]:
from matplotlib.lines import Line2D

def evaluate_curve_from_row(row, d_grid):
    t = row["best_fit_type"]
    if t == "Constant Fit":
        return np.full_like(d_grid, row["constant_c"])
    if t == "Piecewise Linear Fit":
        return PiecewiseLinearFit.piecewise_plateau(
            d_grid, row["piecewise_linear_b"], row["piecewise_linear_m"], row["piecewise_linear_c"]
        )
    if t == "Exponential Saturation Fit":
        return ExponentialSaturationFit.exp_sat(
            d_grid, row["exponential_saturation_a"], row["exponential_saturation_b"], row["exponential_saturation_c"]
        )
    if t == "Michaelis-Menten Fit":
        return MichaelisMentenFit.michaelis_menten(
            d_grid, row["michaelis_menten_a"], row["michaelis_menten_b"], row["michaelis_menten_c"]
        )
    raise ValueError(f"unknown fit type: {t}")

def normalized_curve_from_row(row, d_grid):
    y = evaluate_curve_from_row(row, d_grid)
    span = y[-1] - y[0]
    return (y - y[0]) / span if abs(span) > 1e-9 else np.zeros_like(y)

d_grid = np.linspace(0, d_max, 200)

def normalized_piecewise_curve(fit_obj, d_grid):
    b = fit_obj.params["piecewise_linear_b"]
    m = fit_obj.params["piecewise_linear_m"]
    c = fit_obj.params["piecewise_linear_c"]
    y = PiecewiseLinearFit.piecewise_plateau(d_grid, b, m, c)
    span = y[-1] - y[0]
    return (y - y[0]) / span if abs(span) > 1e-9 else np.zeros_like(y)

piecewise_fits_by_strategy = {
    "stage1_total_counts": [
        flow_stage1.best_fits[name] for name in stage1_fit_quality.index
        if stage1_fit_quality.loc[name, "best_fit_type"] == "Piecewise Linear Fit"
    ],
    "stage2_per_gene": [
        flow_stage2.best_fits[g] for g in stage2_fit_quality.index
        if stage2_fit_quality.loc[g, "best_fit_type"] == "Piecewise Linear Fit"
    ],
    "stage3_pca": [flow.best_fits[pc] for pc in elbow_table["pc"]],
}

colors = {"stage1_total_counts": "steelblue", "stage2_per_gene": "darkorange", "stage3_pca": "seagreen"}

fig, axes = plt.subplots(1, 4, figsize=(21, 4.5), sharey=True)
for ax, (strategy, fits) in zip(axes[:3], piecewise_fits_by_strategy.items()):
    curves = [normalized_piecewise_curve(f, d_grid) for f in fits]
    for curve in curves:
        ax.plot(d_grid, curve, color=colors[strategy], alpha=0.15, linewidth=1)
    if curves:
        median_curve = np.median(curves, axis=0)
        ax.plot(d_grid, median_curve, color="black", linewidth=2, label="median")
        ax.legend(loc="lower right", fontsize=8)
    ax.set_title(f"{strategy} (n={len(fits)})")
    ax.set_xlabel("distance to border")

ax4 = axes[3]
mt_rows = all_genes_fit_quality[all_genes_fit_quality["is_mt"]]
non_mt_rows = all_genes_fit_quality[~all_genes_fit_quality["is_mt"]]
for _, row in non_mt_rows.iterrows():
    ax4.plot(d_grid, normalized_curve_from_row(row, d_grid), color="lightgray", alpha=0.05,
              linewidth=0.8, zorder=1, rasterized=True)
for _, row in mt_rows.iterrows():
    ax4.plot(d_grid, normalized_curve_from_row(row, d_grid), color="crimson", alpha=0.8,
              linewidth=1.5, zorder=2)
ax4.legend(handles=[Line2D([0], [0], color="crimson", lw=1.5, label="mitochondrial")],
           loc="lower right", fontsize=8)
ax4.set_title(f"all genes (n={len(all_genes_fit_quality)}), mt highlighted (n={len(mt_rows)})")
ax4.set_xlabel("distance to border")

axes[0].set_ylabel("normalized fitted value\n(0 = at d=0, 1 = at own asymptote)")
plt.tight_layout()
plt.savefig(OUT_DIR / "piecewise_fits_overlay_by_strategy.svg")
plt.savefig(OUT_DIR / "piecewise_fits_overlay_by_strategy.png", dpi=150)
plt.show()
